In [ ]:
# -*- coding: utf-8 -*-
import random
import time
from itertools import product

# =============== CONFIG ===============
R = 10                # rounds for RR-BRD; we compute q_mu(R)
NUM_INSTANCES = 10    # per (k,n)
SEED = 1
KS = [3, 4]
NS = [2, 3]
VALUE_RANGE = (-100, 100)  # inclusive integer range for d^i and D^{-i}
# =====================================

random.seed(SEED)

# ---------- helpers: strategy enumeration ----------
def enumerate_strategies(k):
    """Returns a list of all 2^k binary vectors (as tuples)."""
    return [tuple(bits) for bits in product([0,1], repeat=k)]

def index_map(list_of_things):
    """Returns a dict mapping thing -> index, preserving order."""
    return {x:i for i,x in enumerate(list_of_things)}

def split_profile(profile, i):
    """Split a joint profile into (s_i, s_-i tuple)."""
    return profile[i], tuple(profile[j] for j in range(len(profile)) if j != i)

# ---------- instance generation & utilities ----------
def sample_instance(k, n):
    """
    Builds a random instance:
      - strategies S_i = {0,1}^k
      - for each player i: d^i in Z^{k}, D^{-i} in Z^{|S_i| x |S_{-i}|}
    Returns a dict with all tables & index maps.
    """
    S_i = [enumerate_strategies(k) for _ in range(n)]
    idx_i = [index_map(S_i[p]) for p in range(n)]

    # Build the set S_-i for each i (cartesian product of others)
    S_minus_i = []
    idx_minus_i = []
    for i in range(n):
        others = [S_i[j] for j in range(n) if j != i]
        Si_ = list(product(*others))  # tuples of length n-1, each entry is a k-tuple
        S_minus_i.append(Si_)
        idx_minus_i.append(index_map(Si_))

    # Random parameter generation
    d = []       # list of length n; each is a list length k
    D = []       # list of length n; each is |S_i| x |S_{-i}| table (list of lists)
    for i in range(n):
        di = [random.randint(VALUE_RANGE[0], VALUE_RANGE[1]) for _ in range(k)]
        Di = [[random.randint(VALUE_RANGE[0], VALUE_RANGE[1]) for _ in range(len(S_minus_i[i]))]
              for __ in range(len(S_i[i]))]
        d.append(di)
        D.append(Di)

    return {
        "n": n, "k": k,
        "S_i": S_i, "S_minus_i": S_minus_i,
        "idx_i": idx_i, "idx_minus_i": idx_minus_i,
        "d": d, "D": D
    }

def utility_i(inst, i, profile):
    """u_i(s) = d^i · x^i + D^{-i}[s_i, s_-i]."""
    # unpack
    d_i = inst["d"][i]
    D_i = inst["D"][i]
    idx_i = inst["idx_i"][i]
    idx_minus_i = inst["idx_minus_i"][i]

    s_i, s_minus_i = split_profile(profile, i)
    # dot(d^i, x^i)
    lin = sum(di * xi for di, xi in zip(d_i, s_i))
    # interaction term: pick the scalar D^{-i}[s_i, s_-i]
    r = idx_i[s_i]
    c = idx_minus_i[s_minus_i]
    inter = D_i[r][c]
    return lin + inter

def is_PNE(inst, profile):
    """True iff profile is a (pure) Nash equilibrium."""
    n = inst["n"]
    for i in range(n):
        # find best response value given others fixed
        s_i_current, s_minus_i = split_profile(profile, i)
        best_val = None
        for alt in inst["S_i"][i]:
            trial = list(profile)
            trial[i] = alt
            ui = utility_i(inst, i, tuple(trial))
            if best_val is None or ui > best_val:
                best_val = ui
        # utility at current strategy
        curr_ui = utility_i(inst, i, profile)
        if curr_ui < best_val:
            return False
    return True

def ensure_instance_with_pne(k, n, max_tries=10_000):
    """Resample until at least one PNE exists."""
    for _ in range(max_tries):
        inst = sample_instance(k, n)
        # scan all profiles
        for prof in product(*inst["S_i"]):
            if is_PNE(inst, prof):
                return inst
    raise RuntimeError(f"Failed to find a ({k},{n}) instance with a PNE in {max_tries} tries.")

# ---------- best responses & RR-BRD ----------
def best_response(inst, i, profile):
    """Return one best-response strategy (tie-broken uniformly) for player i."""
    best_val = None
    best_set = []
    for alt in inst["S_i"][i]:
        trial = list(profile)
        trial[i] = alt
        ui = utility_i(inst, i, tuple(trial))
        if best_val is None or ui > best_val:
            best_val = ui
            best_set = [alt]
        elif ui == best_val:
            best_set.append(alt)
    return random.choice(best_set)

# --- replace these functions in your previous script ---

def RR_BRD_one_trial(inst, init_profile, R, fixed_order):
    """
    RR-BRD with a SINGLE random permutation PER initial profile (fixed across rounds).
    End-of-round PNE test only.
    Returns (success_bool, rounds_used, final_profile_ignored_here).
    - rounds_used = the round index at which convergence occurred (1..R) if success
                    = R if not successful within the cap
    """
    current = tuple(init_profile)
    prev_after_round = current

    for r in range(1, R+1):
        # sweep players in the fixed order
        prof = list(prev_after_round)
        for i in fixed_order:
            prof[i] = best_response(inst, i, tuple(prof))
        prof = tuple(prof)

        # end-of-round PNE test
        if prof == prev_after_round:
            # reached a fixed point -> PNE
            return True, r, prof

        prev_after_round = prof

    return False, R, prev_after_round


def q_mu_for_instance(inst, R, seed=None):
    """
    Enumerate ALL initial profiles.
    For each initial, use ONE random permutation (fixed across rounds).
    Returns:
      q_mu(R), successes, total_inits, avg_rounds_success, avg_rounds_all
    where avg_rounds_all counts failures as R (since rounds_used=R on failure).
    """
    if seed is not None:
        random.seed(seed)

    all_profiles = list(product(*inst["S_i"]))
    n = inst["n"]

    successes = 0
    rounds_success = []
    rounds_all = []

    for s0 in all_profiles:
        order = list(range(n))
        random.shuffle(order)
        ok, rounds_used, _ = RR_BRD_one_trial(inst, s0, R, order)
        rounds_all.append(rounds_used)
        if ok:
            successes += 1
            rounds_success.append(rounds_used)

    total = len(all_profiles)
    q = successes / total
    avg_success = (sum(rounds_success) / len(rounds_success)) if rounds_success else None
    avg_R_all = sum(rounds_all) / total if total > 0 else None
    avg_S_all = sum(rounds_all*n) / total if total > 0 else None
    return q, successes, total, avg_success, avg_R_all, avg_S_all



In [ ]:
# ---------- experiment driver ----------
results = []  # rows: dict with (k, n, inst_id, q, successes, total, seconds)
t_all = time.time()
seed_counter = 0

for n in NS:          # sort by n first to match your preference
    for k in KS:
        print(f"\n=== (n={n}, k={k}) ===")
        for inst_id in range(1, NUM_INSTANCES + 1):
            t0 = time.time()
            inst = ensure_instance_with_pne(k, n)
            q, succ, tot, avgR_succ, avgR_all, avgS_all = q_mu_for_instance(inst, R, seed=seed_counter)
            seed_counter += 1
            sec = time.time() - t0
            esb = (n * R / q) if q > 0 else float("nan")

            results.append({
                "n": n, "k": k, "inst": inst_id,
                "q_mu(R)": q,
                "successes": succ, "total_inits": tot,
                "avg_rounds_all": avgR_all,
                # "avg_steps_taken": avgS_all,
                "expected_step_bound": esb,
                "seconds": round(sec, 3),
                # "avg_rounds_success": avgR_succ,
            })

            succ_str = f"{avgR_succ:.2f}" if avgR_succ is not None else "NA"
            esb_str = f"{esb:.2f}" if q > 0 else "inf"
            print(f"instance {inst_id:02d}: q_mu(R)={q:.4f}  "
                  f"(succ={succ}/{tot}, avgR_all={avgR_all:.2f}, avgS_all={avgS_all:.2f}, ESB={esb_str}, {sec:.2f}s)")

print(f"\nTotal wall time: {time.time() - t_all:.2f}s")


# summarize per (k,n)
print("\n====== SUMMARY ======")
for n in NS:
    for k in KS:
        rows = [r for r in results if r["k"] == k and r["n"] == n]
        if not rows: continue
        cnt_pos = sum(1 for r in rows if r["q_mu(R)"] > 0.0)
        avg_q = sum(r["q_mu(R)"] for r in rows) / len(rows)
        avgR_all = sum(r["avg_rounds_all"] for r in rows) / len(rows)
        # avgS_all = sum(r["avg_steps_taken"] for r in rows) / len(rows)
        avg_esb = sum(r["expected_step_bound"] for r in rows) / len(rows) 
        # avgR_succ = sum(r["avg_rounds_success"] for r in rows) / len(rows)
        print(f"(k={k}, n={n})  instances={len(rows)}, "
              f"#(q_mu(R)>0)={cnt_pos}/{len(rows)}, "
              f"avg q_mu(R)={avg_q:.4f}",
                f"avg_R = {avgR_all}",
                # f"avg_S = {avgS_all}",
                f"avg_esb = {avg_esb}",
                # f"avg_R_suc = {avgR_succ}"
        )

print(f"\nTotal wall time: {time.time() - t_all:.2f}s")


In [ ]:
# ---- Simple CSV export (sorted by n, then k) ----
import pandas as pd
from pathlib import Path

out_dir = Path("./nf_qmu_outputs")
out_dir.mkdir(exist_ok=True)

# Per-instance CSV (sort by n, k, then inst)
df = pd.DataFrame(results).sort_values(["n", "k", "inst"]).reset_index(drop=True)
per_instance_path = out_dir / f"nf_qmu_instances_R{R}.csv"
df.to_csv(per_instance_path, index=False)

# Per-(n,k) summary CSV (group and sort by n, then k)
summary = (
    df.groupby(["n","k"], as_index=False)
      .agg(
          num_instances=("inst", "count"),
          num_with_q_pos=("q_mu(R)", lambda s: (s > 0).sum()),
          avg_q_mu_R=("q_mu(R)", "mean"),
          avg_rounds_all_over_instances=("avg_rounds_all", "mean"),
          # avg_steps_taken_all_over_instances=("avg_steps_taken", "mean"),
          avg_esb=("expected_step_bound", "mean"),
          # avg_rounds_success_over_instances=("avg_rounds_success", lambda s: s.dropna().mean()),
      )
      .sort_values(["n","k"])
      .reset_index(drop=True)
)
summary_path = out_dir / f"nf_qmu_summary_R{R}.csv"
summary.to_csv(summary_path, index=False)

print(f"\nCSV saved:\n  - Per-instance: {per_instance_path}\n  - Summary:      {summary_path}")
